# Bài Thực Hành Chương 6: Xử Lý & Biến Đổi Dữ Liệu Nâng Cao với Pandas

### Mục tiêu học tập (CLO2, CLO4, CLO5):
- Nạp dữ liệu từ CSV và SQLite qua Pandas (`read_csv`, `read_sql`).
- Xử lý dữ liệu thiếu (`isna`, `fillna`, `dropna`).
- Trích xuất và lọc dữ liệu nâng cao (`loc`, `iloc`, boolean indexing, `query`).
- Nối ghép nhiều bảng dữ liệu quan hệ (`merge`, `concat`).
- Thao tác chuỗi thời gian (`to_datetime`, `.dt`, `to_period`, `rolling`).
- Xây dựng Ma trận Cohort Retention bằng `pivot_table`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loaders.csv_loader import CSVDataLoader
from src.processing.cleaner import DataCleaner
from src.processing.merger import DataMerger

loader = CSVDataLoader(project_root / 'data' / 'raw')
raw_data = loader.load()


## 1. Kiểm Tra & Xử Lý Dữ Liệu Khuyết (Missing Data)


In [ ]:
print('--- Tỷ lệ giá trị thiếu (Missing Rate %) trên bảng orders ---')
print(raw_data['orders'].isna().mean() * 100)

print('\n--- Tỷ lệ giá trị thiếu trên bảng refunds ---')
print(raw_data['refunds'].isna().mean() * 100)

# Làm sạch dữ liệu thông qua DataCleaner
cleaner = DataCleaner(raw_data)
clean_data = cleaner.clean_all()
print('\nLàm sạch hoàn tất! Số dòng orders hợp lệ:', len(clean_data['orders']))


## 2. Nối Ghép Bảng Đa Chiều & Tính Doanh Thu Thuần (Net Sales)


In [ ]:
merger = DataMerger(clean_data)
master_orders = merger.build_master_orders()

print(master_orders[['order_id', 'customer_id', 'order_status', 'subtotal', 'paid_amount', 'refunded_amount', 'net_sales']].head())


## 3. Thao Tác Chuỗi Thời Gian & Rolling Moving Average (Chapter 6.5)


In [ ]:
# Tổng hợp doanh số theo tháng
monthly_ts = master_orders.groupby('order_month').agg(
    orders_count=('order_id', 'count'),
    net_revenue=('net_sales', 'sum')
).reset_index()

# Tính trung bình động 3 tháng (3-Month Rolling Average)
monthly_ts['rolling_3m_avg'] = monthly_ts['net_revenue'].rolling(window=3, min_periods=1).mean().round(2)
print(monthly_ts.head(10))


## 4. Xây Dựng Ma Trận Phân Tích Cohort Retention


In [ ]:
from src.processing.feature_engineering import FeatureEngineer

items = merger.build_master_order_items(master_orders)
engineer = FeatureEngineer(master_orders, items)
cohort_counts, retention_pct = engineer.compute_cohort_retention_matrix()

print('--- Ma trận Tỷ Lệ Giữ Chân Khách Hàng (%) (6 Cohort đầu tiên) ---')
print(retention_pct.iloc[:6, :8].round(2))
